# BN3406 Group Project — Diabetic Retinopathy Image Synthesis
## Conditional DCGAN for Fundus Image Generation (Grades 3 & 4)

**Dataset:** Kaggle Diabetic Retinopathy Detection (EyePacs)  
**Goal:** Train a Conditional GAN to generate synthetic high-grade DR fundus images, evaluate with FID, and test whether augmenting a classifier with synthetic data improves grading performance.

---

## 1. Background & Motivation

### Why Generative AI for Medical Imaging?

This project is directly motivated by the three core problems discussed in **Week 6 (Image Generation/Synthesis)**:

| Problem | How it applies to our project |
|---|---|
| **Data Problem** — deep learning is data-hungry, but medical data is scarce and expensive | Kaggle DR dataset is heavily skewed: Grade 0 has ~25k images, Grade 4 has ~700. We use GANs to synthesise the minority classes. |
| **Acquisition Problem** — high-quality scans are slow or harmful | Fundus photography requires clinical equipment and trained ophthalmologists. Generating synthetic images bypasses this bottleneck. |
| **Modality Problem** — patients need multiple modalities but machines aren't always available | Not directly applicable here, but related: generative models can synthesise imaging data from underrepresented populations. |

The lecture framed this as a **conditional generation** task — specifically the `p(x | c)` formulation where:
- `x` = fundus image
- `c` = DR severity grade (0–4, from the ETDRS scale)

This puts our project squarely in the **Conditional Generative** category from the lecture's taxonomy (slide 16).

### Why a GAN rather than a VAE or Diffusion Model?

The lecture explicitly compared these approaches (slides 41–42):
- **GANs** produce **sharp, realistic images** but have **unstable training** (mode collapse risk)
- **VAEs** have **stable training** but produce **blurry outputs** — not ideal for medical images where fine lesion detail matters
- **Diffusion models** produce the best quality today but require significantly more compute and training time

For a course project with limited compute, a **Conditional DCGAN** is the most practical choice that still produces visually informative outputs. The training instability is actually a useful negative result to discuss if it occurs.

### The Conditional GAN Objective

From slide 55, the cGAN loss extends the standard GAN minimax game by conditioning both G and D on label `c`:

$$\min_G \max_D \; \mathbb{E}_{x \sim p_{data}(x)}[\log D(x, c)] + \mathbb{E}_{z \sim p(z)}[\log(1 - D(G(z, c), c))]$$

- **D-step**: fix G, train D to push `D(real, c) → 1` and `D(fake, c) → 0`
- **G-step**: fix D, train G to push `D(G(z, c), c) → 1` (confuse D)

The label enters the model via **Input Concatenation** (for G) and **label broadcast as extra channel** (for D) — both are methods described in slide 57.

## 2. Setup & Imports

In [ ]:
import os
import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image
from PIL import ImageFile
ImageFile.LOAD_TRUNCATED_IMAGES = True
from tqdm import tqdm

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms as transforms
import torchvision.utils as vutils
from torchvision import models

print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")

In [ ]:
# ── Reproducibility ─────────────────────────────────────────────────────────
# Setting seeds so results are consistent across runs. And for reproducibility.

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.backends.cudnn.deterministic = True

# ── Config ──────────────────────────────────────────────────────────────────
# Centralise all hyperparameters here for easy tuning and reference.

CONFIG = {
    "data_dir":      "./kaggle_dr",
    "output_dir":    "./outputs_e20-s200",
    "epochs":        25,
    "batch_size":    10,           # Small batch size
    "lr":            0.0002,       # Standard DCGAN learning rate
    "beta1":         0.5,          # Adam beta1 — paper recommends 0.5 for GANs
    "latent_dim":    64,           # Size of noise vector z
    "img_size":      128,          # Resize images
    "n_classes":     5,            # DR grades 0, 1, 2, 3, 4
    "target_grades": [3, 4],       # Grades we want to over-sample with synthetic data
    "n_generate":    100,          # Synthetic images per target grade
    "fid_batch":     32,
    "save_every":    10,
    "sample_every":  5,
    "num_workers":   0,
}

# Auto-select device, i dont have a compatible gpu :(
if torch.cuda.is_available():
    device = torch.device("cuda")
elif torch.backends.mps.is_available():
    device = torch.device("mps")
else:
    device = torch.device("cpu")

print(f"Using device: {device}")

os.makedirs(CONFIG["output_dir"], exist_ok=True)
os.makedirs(os.path.join(CONFIG["output_dir"], "samples"), exist_ok=True)
os.makedirs(os.path.join(CONFIG["output_dir"], "checkpoints"), exist_ok=True)

## 3. Dataset — Kaggle DR

### Dataset Overview

The [Kaggle Diabetic Retinopathy Detection dataset](https://www.kaggle.com/c/diabetic-retinopathy-detection) contains ~88,000 colour fundus photographs labelled on the **ETDRS scale (0–4)**:

| Grade | Severity | Description |
|---|---|---|
| 0 | No DR | Healthy retina |
| 1 | Mild | Small microaneurysms |
| 2 | Moderate | More extensive lesions, some haemorrhages |
| 3 | Severe | Many haemorrhages, venous beading |
| 4 | Proliferative | Neovascularisation, most advanced stage |

### The Class Imbalance Problem

This is a classic **Data Problem** (lecture slide 10). Grade 3 and 4 are severely under-represented, which makes training any supervised model difficult — the model will bias toward predicting Grade 0 because that minimises overall loss. This is the core clinical motivation for using a GAN to synthesise more Grade 3 and 4 images.

### Folder structure expected:
```
kaggle_dr/
 train_1/ ← .jpeg fundus images
 train_2/
 ...
 train_5/
 
 test_1/
 test_2/
 ...
 test_7/
 trainLabels.csv ← columns: image, level

In [ ]:
import cv2

# ── Preprocessing ────────────────────────────────────────────────────────────
# Applied to every image at load time inside KaggleDRDataset.

CLAHE_CLIP_LIMIT = 2.0  # Clip limit: lower = subtler, higher = more aggressive
CLAHE_TILE_SIZE  = 8    # Local tile size (8x8 is standard for fundus images)

def standardise_brightness(img: Image.Image) -> Image.Image:
    """
    Standardise image brightness using CLAHE on the V channel (HSV).

    CLAHE (Contrast Limited Adaptive Histogram Equalisation) equalises
    brightness locally in small tiles, so the large black border around
    the fundus does not skew the result the way global equalisation would.
    The clip limit prevents noise amplification in flat regions.
    Hue and saturation are unchanged.
    """
    clahe   = cv2.createCLAHE(clipLimit=CLAHE_CLIP_LIMIT,
                               tileGridSize=(CLAHE_TILE_SIZE, CLAHE_TILE_SIZE))
    img_hsv = cv2.cvtColor(np.array(img), cv2.COLOR_RGB2HSV)
    img_hsv[:, :, 2] = clahe.apply(img_hsv[:, :, 2])
    return Image.fromarray(cv2.cvtColor(img_hsv, cv2.COLOR_HSV2RGB))


# ── Visual sanity check: one sample per grade, before vs after ────────────

n_grades = 5
fig, axes = plt.subplots(2, n_grades, figsize=(15, 6))

for grade in range(n_grades):
    img_name = df_csv[df_csv["level"] == grade].iloc[0]["image"]
    raw_img  = Image.open(image_paths[img_name]).convert("RGB")
    proc_img = standardise_brightness(raw_img)

    for row, img in enumerate([raw_img, proc_img]):
        axes[row, grade].imshow(img.resize((CONFIG["img_size"], CONFIG["img_size"])))
        axes[row, grade].axis("off")
        if row == 0:
            axes[row, grade].set_title(f"Grade {grade}", fontsize=10)

axes[0, 0].set_ylabel("Original",    fontsize=10, labelpad=8)
axes[1, 0].set_ylabel("CLAHE",       fontsize=10, labelpad=8)

fig.suptitle("Brightness Standardisation via CLAHE (per DR Grade)", fontsize=12)
plt.tight_layout()
plt.show()

print("standardise_brightness defined — will be applied inside KaggleDRDataset.__getitem__")

In [ ]:
from torchvision import transforms
from torch.utils.data import DataLoader, Subset
import numpy as np

# ── Transforms ─────────────────────────────────────────────
def get_transforms(img_size):
    train_tf = transforms.Compose([
        transforms.Resize((img_size, img_size)),
        transforms.RandomHorizontalFlip(),
        transforms.RandomVerticalFlip(),
        transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.1),
        transforms.ToTensor(),
        transforms.Normalize([0.5, 0.5, 0.5], [0.5, 0.5, 0.5]),
    ])

    eval_tf = transforms.Compose([
        transforms.Resize((img_size, img_size)),
        transforms.ToTensor(),
        transforms.Normalize([0.5, 0.5, 0.5], [0.5, 0.5, 0.5]),
    ])

    return train_tf, eval_tf


# ── Load dataset ───────────────────────────────────────────
train_tf, eval_tf = get_transforms(CONFIG["img_size"])
dataset = KaggleDRDataset(CONFIG["data_dir"], transform=train_tf)

# ── Balanced subsampling: n images per class ─────────────
np.random.seed(42)

labels = np.array(dataset.labels)
indices = []
samples_per_class = 300

for grade in range(CONFIG["n_classes"]):
    grade_indices = np.where(labels == grade)[0]

    n = min(samples_per_class, len(grade_indices))  # handle rare classes safely
    selected = np.random.choice(grade_indices, n, replace=False)

    indices.extend(selected)

indices = np.array(indices)
np.random.shuffle(indices)

dataset = Subset(dataset, indices)

print(f"Using {len(dataset)} images ({samples_per_class} per class where available)")

# ── DataLoader ─────────────────────────────────────────────
dataloader = DataLoader(
    dataset,
    batch_size=4,
    shuffle=True,
    num_workers=0,   # keep 0 while debugging
    pin_memory=True
)

print(f"Total images: {len(dataset)}")

In [ ]:
# ── Grade distribution visualisation ────────────────────────────────────────

# Handle both full dataset and Subset
if isinstance(dataset, torch.utils.data.Subset):
    labels_used = dataset.dataset.labels[dataset.indices]
else:
    labels_used = dataset.labels

labels_arr, counts = np.unique(labels_used, return_counts=True)

grade_names = ["Grade 0\n(No DR)", "Grade 1\n(Mild)", "Grade 2\n(Moderate)",
               "Grade 3\n(Severe)", "Grade 4\n(Proliferative)"]
colours = ["#2A9D8F", "#57CC99", "#E9C46A", "#F4A261", "#E63946"]

fig, ax = plt.subplots(figsize=(9, 4))
bars = ax.bar(grade_names, counts, color=colours, edgecolor="white", linewidth=0.8)

for bar, cnt in zip(bars, counts):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 50,
            f"{cnt:,}", ha="center", va="bottom", fontsize=10, fontweight="bold")

ax.set_title("Kaggle DR Dataset — Class Imbalance Problem", fontsize=13, pad=12)
ax.set_ylabel("Number of Images")
ax.spines[["top", "right"]].set_visible(False)
ax.set_ylim(0, max(counts) * 1.15)

# Annotate the target grades
for i in [3, 4]:
    ax.annotate("← We generate\nsynthetic data\nfor these",
                xy=(i, counts[i]),
                xytext=(i + 0.5, counts[i] + max(counts)*0.1),
                arrowprops=dict(arrowstyle="->", color="#E63946"),
                color="#E63946", fontsize=8, ha="left")

plt.tight_layout()
plt.savefig(os.path.join(CONFIG["output_dir"], "grade_distribution.png"), dpi=150)
plt.show()

In [ ]:
# ── Preview a few real fundus images ───────────────────────────────────────

preview_tf = transforms.Compose([
    transforms.Resize((CONFIG["img_size"], CONFIG["img_size"])),
    transforms.ToTensor(),
])

fig, axes = plt.subplots(1, 5, figsize=(15, 3))
grade_labels_text = ["Grade 0", "Grade 1", "Grade 2", "Grade 3", "Grade 4"]

df_csv = pd.read_csv(os.path.join(CONFIG["data_dir"], "trainLabels.csv"))

# 🔥 Build image lookup (FIXED: only include real directories)
image_paths = {}
train_folders = [
    os.path.join(CONFIG["data_dir"], d)
    for d in os.listdir(CONFIG["data_dir"])
    if d.startswith("train") and os.path.isdir(os.path.join(CONFIG["data_dir"], d))
]

print(f"Found {len(train_folders)} train folders")

for folder in train_folders:
    for fname in os.listdir(folder):
        if fname.endswith(".jpeg"):
            key = fname.replace(".jpeg", "")
            image_paths[key] = os.path.join(folder, fname)

print(f"Indexed {len(image_paths)} images")

# Display one sample per grade
for grade in range(5):
    row = df_csv[df_csv["level"] == grade].iloc[0]
    img_name = row["image"]

    if img_name not in image_paths:
        raise FileNotFoundError(f"{img_name}.jpeg not found")

    img = Image.open(image_paths[img_name]).convert("RGB")
    img = img.resize((CONFIG["img_size"], CONFIG["img_size"]))

    axes[grade].imshow(img)
    axes[grade].set_title(grade_labels_text[grade], fontsize=10)
    axes[grade].axis("off")

fig.suptitle("Sample Real Fundus Images by DR Grade", fontsize=12, y=1.02)
plt.tight_layout()
plt.show()

### 3b. Image Preprocessing — Brightness Standardisation

Fundus photographs in the Kaggle DR dataset come from multiple clinical sites with different cameras and lighting conditions, so raw pixel brightness varies considerably across images. This inconsistency makes GAN training harder — the discriminator has to distinguish real from fake *and* deal with arbitrary exposure differences.

We standardise brightness using **CLAHE** (Contrast Limited Adaptive Histogram Equalisation), applied to the V (brightness) channel in HSV colour space:

- Works on small local tiles rather than the whole image, so the large black border around the fundus doesn't skew the result
- A clip limit caps contrast amplification to avoid over-enhancing noise
- Hue and saturation are left untouched, preserving the colour structure of the retina

This is applied **at load time** inside `KaggleDRDataset` so the raw files on disk are never modified.

In [ ]:
import cv2

# ── Preprocessing ────────────────────────────────────────────────────────────
# Applied to every image at load time inside KaggleDRDataset.

CLAHE_CLIP_LIMIT = 2.0  # Clip limit: lower = subtler, higher = more aggressive
CLAHE_TILE_SIZE  = 8    # Local tile size (8x8 is standard for fundus images)

def standardise_brightness(img: Image.Image) -> Image.Image:
    """
    Standardise image brightness using CLAHE on the V channel (HSV).

    CLAHE (Contrast Limited Adaptive Histogram Equalisation) equalises
    brightness locally in small tiles, so the large black border around
    the fundus does not skew the result the way global equalisation would.
    The clip limit prevents noise amplification in flat regions.
    Hue and saturation are unchanged.
    """
    clahe   = cv2.createCLAHE(clipLimit=CLAHE_CLIP_LIMIT,
                               tileGridSize=(CLAHE_TILE_SIZE, CLAHE_TILE_SIZE))
    img_hsv = cv2.cvtColor(np.array(img), cv2.COLOR_RGB2HSV)
    img_hsv[:, :, 2] = clahe.apply(img_hsv[:, :, 2])
    return Image.fromarray(cv2.cvtColor(img_hsv, cv2.COLOR_HSV2RGB))


# ── Visual sanity check: one sample per grade, before vs after ───────────────

n_grades = 5
fig, axes = plt.subplots(2, n_grades, figsize=(15, 6))

for grade in range(n_grades):
    img_name = df_csv[df_csv["level"] == grade].iloc[0]["image"]
    raw_img  = Image.open(image_paths[img_name]).convert("RGB")
    proc_img = standardise_brightness(raw_img)

    for row, display_img in enumerate([raw_img, proc_img]):
        axes[row, grade].imshow(display_img.resize((CONFIG["img_size"], CONFIG["img_size"])))
        axes[row, grade].axis("off")
        if row == 0:
            axes[row, grade].set_title(f"Grade {grade}", fontsize=10)

axes[0, 0].set_ylabel("Original",   fontsize=10, labelpad=8)
axes[1, 0].set_ylabel("CLAHE",      fontsize=10, labelpad=8)

fig.suptitle("Brightness Standardisation via CLAHE (per DR Grade)", fontsize=12)
plt.tight_layout()
plt.show()

print("standardise_brightness defined — will be applied inside KaggleDRDataset.__getitem__")

## 4. Model Architecture — Conditional DCGAN

### Generative Model Taxonomy

From the lecture taxonomy (slide 21), we are implementing an **Implicit Density / Adversarial (GAN)** model. Unlike VAEs (which explicitly model `p(z|x)` and `p(x|z)` through an ELBO objective), GANs use **implicit density estimation** — we never explicitly write down a probability distribution. Instead, we train two networks that compete:

- **Generator G**: maps noise `z ~ p_z` → fake image (learns to fool D)
- **Discriminator D**: classifies images as real or fake (binary classification)

The GAN minimax game is:
$$\min_G \max_D \; \mathbb{E}_{x \sim p_{data}}[\log D(x)] + \mathbb{E}_{z \sim p_z}[\log(1 - D(G(z)))]$$

### Making it Conditional

A standard GAN samples from `p(x)` — we have no control over what grade gets generated. By conditioning on grade label `c`, we switch to `p(x | c)`, giving us the ability to specifically request Grade 3 or Grade 4 images.

The cGAN objective (Mirza & Osindero, 2014 — slide 55):
$$\min_G \max_D \; \mathbb{E}_{x \sim p_{data}(x)}[\log D(x, c)] + \mathbb{E}_{z \sim p(z)}[\log(1 - D(G(z, c), c))]$$

### How We Inject the Condition

Slide 57 shows four ways to inject condition `c` into the model. We use two:

1. **Generator — Input Concatenation**: embed the grade label into a vector and concatenate it with noise `z` before feeding into the first layer
2. **Discriminator — Extra channel**: project the grade label to an `H×W` map and concatenate it as an additional channel to the input image

In [ ]:
def weights_init(m):
    """
    Initialise weights from the DCGAN paper (Radford et al., 2015).
    Conv weights: N(0, 0.02), BatchNorm weights: N(1, 0.02), biases: 0.
    This specific initialisation is important for stable GAN training.
    """
    classname = m.__class__.__name__
    if "Conv" in classname:
        nn.init.normal_(m.weight.data, 0.0, 0.02)
    elif "BatchNorm" in classname:
        nn.init.normal_(m.weight.data, 1.0, 0.02)
        nn.init.constant_(m.bias.data, 0)

In [ ]:
class Generator(nn.Module):
    """
    Conditional Generator: noise z + grade label c → 128x128 RGB fundus image.
    
    Architecture:
    - Grade label is embedded (learnable) and concatenated with z
    - 5 transposed convolution blocks, each doubling spatial resolution
    - 1x1 → 4x4 → 8x8 → 16x16 → 32x32 → 64x64 → 128x128
    - BatchNorm + ReLU after each block, Tanh at the end (output in [-1, 1])
    
    This maps to the 'Input Concatenation' conditioning strategy from
    slide 57 — the condition c is concatenated with input z.
    """
    def __init__(self, latent_dim, n_classes, img_size, ngf=64):
        super().__init__()
        self.label_emb = nn.Embedding(n_classes, n_classes)
        input_dim = latent_dim + n_classes

        self.net = nn.Sequential(
            nn.ConvTranspose2d(input_dim, ngf * 16, 4, 1, 0, bias=False),
            nn.BatchNorm2d(ngf * 16),
            nn.ReLU(True),
            # → ngf*16 × 4 × 4

            nn.ConvTranspose2d(ngf * 16, ngf * 8, 4, 2, 1, bias=False),
            nn.BatchNorm2d(ngf * 8),
            nn.ReLU(True),
            # → ngf*8 × 8 × 8

            nn.ConvTranspose2d(ngf * 8, ngf * 4, 4, 2, 1, bias=False),
            nn.BatchNorm2d(ngf * 4),
            nn.ReLU(True),
            # → ngf*4 × 16 × 16

            nn.ConvTranspose2d(ngf * 4, ngf * 2, 4, 2, 1, bias=False),
            nn.BatchNorm2d(ngf * 2),
            nn.ReLU(True),
            # → ngf*2 × 32 × 32

            nn.ConvTranspose2d(ngf * 2, ngf, 4, 2, 1, bias=False),
            nn.BatchNorm2d(ngf),
            nn.ReLU(True),
            # → ngf × 64 × 64

            nn.ConvTranspose2d(ngf, 3, 4, 2, 1, bias=False),
            nn.Tanh()
            # → 3 × 128 × 128
        )

    def forward(self, z, labels):
        label_input = self.label_emb(labels)                       # (B, n_classes)
        gen_input   = torch.cat([z, label_input], dim=1)           # (B, latent+n_classes)
        gen_input   = gen_input.unsqueeze(2).unsqueeze(3)           # (B, C, 1, 1)
        return self.net(gen_input)

In [ ]:
class Discriminator(nn.Module):
    """
    Conditional Discriminator: (image, grade label) → real/fake probability.

    Architecture:
    - Grade label is embedded to a small 16×16 map, then bilinearly upsampled
      to match img_size — resolution-agnostic and memory-efficient
    - 5 strided convolution blocks, each halving spatial resolution
    - LeakyReLU throughout, Sigmoid at the end
    """
    def __init__(self, n_classes, img_size, ndf=64):
        super().__init__()
        self.img_size  = img_size
        self.label_emb = nn.Embedding(n_classes, 16 * 16)  # fixed small map

        self.net = nn.Sequential(
            # Input: 4 channels (3 RGB + 1 upsampled label map)
            nn.Conv2d(4, ndf, 4, 2, 1, bias=False),
            nn.LeakyReLU(0.2, inplace=True),

            nn.Conv2d(ndf, ndf * 2, 4, 2, 1, bias=False),
            nn.BatchNorm2d(ndf * 2),
            nn.LeakyReLU(0.2, inplace=True),

            nn.Conv2d(ndf * 2, ndf * 4, 4, 2, 1, bias=False),
            nn.BatchNorm2d(ndf * 4),
            nn.LeakyReLU(0.2, inplace=True),

            nn.Conv2d(ndf * 4, ndf * 8, 4, 2, 1, bias=False),
            nn.BatchNorm2d(ndf * 8),
            nn.LeakyReLU(0.2, inplace=True),

            nn.Conv2d(ndf * 8, ndf * 16, 4, 2, 1, bias=False),
            nn.BatchNorm2d(ndf * 16),
            nn.LeakyReLU(0.2, inplace=True),

            nn.AdaptiveAvgPool2d((4, 4)),
            nn.Conv2d(ndf * 16, 1, 3, 1, 1, bias=False),
            nn.Sigmoid()
        )

    def forward(self, img, labels):
        label_map = self.label_emb(labels)                    # (B, 256)
        label_map = label_map.view(img.size(0), 1, 16, 16)   # (B, 1, 16, 16)
        label_map = torch.nn.functional.interpolate(
            label_map, size=(img.size(2), img.size(3)),
            mode='bilinear', align_corners=False
        )                                                      # (B, 1, H, W)
        d_input = torch.cat([img, label_map], dim=1)          # (B, 4, H, W)
        return self.net(d_input).view(img.size(0), -1).mean(dim=1)

In [ ]:
# Initialise models
G = Generator(CONFIG["latent_dim"], CONFIG["n_classes"], CONFIG["img_size"]).to(device)
D = Discriminator(CONFIG["n_classes"], CONFIG["img_size"]).to(device)
G.apply(weights_init)
D.apply(weights_init)

n_params_G = sum(p.numel() for p in G.parameters())
n_params_D = sum(p.numel() for p in D.parameters())
print(f"Generator     — {n_params_G:,} parameters")
print(f"Discriminator — {n_params_D:,} parameters")
print(f"Total         — {n_params_G + n_params_D:,} parameters")

## 5. Training the GAN

### The Adversarial Training Loop

Training alternates between two steps, as shown in slides 37–40:

**D-step** (fix G, update D — binary classification):
$$\max_D \; \mathbb{E}[\log D(x, c)] + \mathbb{E}[\log(1 - D(G(z, c), c))]$$
- Push `D(real, c) → 1` (real images should score high)
- Push `D(fake, c) → 0` (fake images should score low)

**G-step** (fix D, update G — fool D):
$$\min_G \; \mathbb{E}[\log(1 - D(G(z, c), c))]$$
- Push `D(G(z, c), c) → 1` (make fake images look real to D)

### Label Smoothing

We use **one-sided label smoothing**: instead of real targets = 1.0, we use 0.9. This prevents D from becoming overconfident, which typically leads to vanishing gradients for G — one of the classic GAN training instabilities.

### What to Watch For

- **Healthy training**: D loss and G loss should both stay non-zero and relatively stable. D loss around 1.0–1.4, G loss around 1.0–3.0 is normal.
- **Mode collapse**: G loss drops to near-zero and all generated images look identical — G found a single image that fools D.
- **D dominates**: D loss → 0, G loss → very high. D became too powerful, G gets no gradient signal.

These are exactly the instability issues the lecture highlighted as GAN's main weakness (slide 42).

In [ ]:
def train_gan(CONFIG, device, dataloader, G, D, opt_G, opt_D, criterion):
    G.train()
    D.train()

    g_losses, d_losses = [], []
    best_g_loss = float("inf")
    checkpoint_dir = os.path.join(CONFIG["output_dir"], "checkpoints")

    print("Testing dataloader...")
    for batch_idx, (imgs, labels) in enumerate(dataloader):
        print(f"Got batch {batch_idx}")
        break

    print(f"Starting GAN training...")
    print(f"Epochs: {CONFIG['epochs']} | Batches per epoch: {len(dataloader)}")

    for epoch in range(1, CONFIG["epochs"] + 1):
        print(f"\n--- Epoch {epoch}/{CONFIG['epochs']} START ---")
        epoch_g = epoch_d = 0.0

        with tqdm(dataloader, desc=f"Epoch {epoch}/{CONFIG['epochs']}") as pbar:
            for batch_idx, (imgs, labels) in enumerate(pbar):
                if batch_idx == 0:
                    print(f"  First batch loaded (epoch {epoch})")

                imgs   = imgs.to(device)
                labels = labels.to(device)
                bs     = imgs.size(0)

                real_targets = torch.full((bs,), 0.9, device=device)
                fake_targets = torch.zeros(bs, device=device)

                # Dstep
                opt_D.zero_grad()
                d_real      = D(imgs, labels)
                loss_d_real = criterion(d_real, real_targets)

                z           = torch.randn(bs, CONFIG["latent_dim"], device=device)
                fake_imgs   = G(z, labels).detach()
                d_fake      = D(fake_imgs, labels)
                loss_d_fake = criterion(d_fake, fake_targets)

                loss_D = loss_d_real + loss_d_fake
                loss_D.backward()
                opt_D.step()

                # Gstep
                opt_G.zero_grad()
                z         = torch.randn(bs, CONFIG["latent_dim"], device=device)
                fake_imgs = G(z, labels)
                d_fake    = D(fake_imgs, labels)
                loss_G    = criterion(d_fake, torch.ones(bs, device=device))
                loss_G.backward()
                opt_G.step()

                epoch_g += loss_G.item()
                epoch_d += loss_D.item()

                if batch_idx % 10 == 0:
                    print(f"    batch {batch_idx}/{len(dataloader)} | D: {loss_D.item():.3f} | G: {loss_G.item():.3f}")

                pbar.set_postfix({"D": f"{loss_D.item():.3f}", "G": f"{loss_G.item():.3f}"})

        print(f"Epoch {epoch} done.")
        g_losses.append(epoch_g / len(dataloader))
        d_losses.append(epoch_d / len(dataloader))
        current_g_loss = g_losses[-1]
        print(f"[Epoch {epoch}] avg D: {d_losses[-1]:.4f} | avg G: {current_g_loss:.4f}")

        # checkpoint
        if epoch % CONFIG["save_every"] == 0 or epoch == CONFIG["epochs"]:
            ckpt_path = os.path.join(checkpoint_dir, f"ckpt_epoch_{epoch:04d}.pt")
            torch.save({
                "epoch":       epoch,
                "G_state":     G.state_dict(),
                "D_state":     D.state_dict(),
                "opt_G_state": opt_G.state_dict(),
                "opt_D_state": opt_D.state_dict(),
                "g_losses":    g_losses,
                "d_losses":    d_losses,
                "CONFIG":      CONFIG,
            }, ckpt_path)
            print(f"  ✓ Checkpoint saved → {ckpt_path}")

        # Bestmodel
        if current_g_loss < best_g_loss:
            best_g_loss = current_g_loss
            best_path = os.path.join(checkpoint_dir, "best_generator.pt")
            torch.save({
                "epoch":   epoch,
                "G_state": G.state_dict(),
                "g_loss":  best_g_loss,
                "CONFIG":  CONFIG,
            }, best_path)
            print(f"  ✓ Best model updated (G loss: {best_g_loss:.4f}) → {best_path}")

    return g_losses, d_losses

In [ ]:
# Optimisers — Adam with beta1=0.5 is standard for GANs (from DCGAN paper)
# Lower beta1 reduces momentum, which helps with the oscillatory nature of
# adversarial training.
opt_G = optim.Adam(G.parameters(), lr=CONFIG["lr"], betas=(CONFIG["beta1"], 0.999))
opt_D = optim.Adam(D.parameters(), lr=CONFIG["lr"], betas=(CONFIG["beta1"], 0.999))

# Binary Cross Entropy — the natural loss for the real/fake binary classification
criterion = nn.BCELoss()

print("Starting training...")
g_losses, d_losses = train_gan(CONFIG, device, dataloader, G, D, opt_G, opt_D, criterion)

In [ ]:
# ── Plot training losses ────────────────────────────────────────────────────
# A healthy GAN shows both losses fluctuating without either collapsing to 0.
# If D loss → 0, mode collapse has likely occurred.

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 4))

ax1.plot(g_losses, color="#E63946", linewidth=1.5, label="Generator Loss")
ax1.plot(d_losses, color="#457B9D", linewidth=1.5, label="Discriminator Loss")
ax1.set_xlabel("Epoch"); ax1.set_ylabel("Loss")
ax1.set_title("GAN Training Losses")
ax1.legend(); ax1.grid(alpha=0.3)
ax1.spines[["top", "right"]].set_visible(False)


# Rolling average to smooth out noise
window = min(10, len(g_losses))
g_smooth = np.convolve(g_losses, np.ones(window)/window, mode="valid")
d_smooth = np.convolve(d_losses, np.ones(window)/window, mode="valid")
ax2.plot(g_smooth, color="#E63946", linewidth=2, label="G (smoothed)")
ax2.plot(d_smooth, color="#457B9D", linewidth=2, label="D (smoothed)")
ax2.set_xlabel("Epoch"); ax2.set_ylabel("Loss")
ax2.set_title(f"Smoothed Losses (window={window})")
ax2.legend(); ax2.grid(alpha=0.3)
ax2.spines[["top", "right"]].set_visible(False)


plt.tight_layout()
plt.savefig(os.path.join(CONFIG["output_dir"], "training_losses.png"), dpi=150)
plt.show()

## 6. Generate Synthetic Images

Once training is done, we use the Generator in **inference mode** to produce new fundus images by:
1. Sampling random noise `z ~ N(0, I)` 
2. Pairing each `z` with a target grade label (3 or 4)
3. Running `G(z, c)` to get a synthetic image

This directly demonstrates the **Conditional Generative** paradigm from slide 16 — the label `c` acts as a control signal that tells G what to generate.

In [ ]:
def generate_synthetic_images(G, CONFIG, device):
    """
    Sample from the trained generator conditioned on target DR grades.
    Saves images to output_dir/synthetic/grade_X/
    """
    G.eval()
    print("Generating synthetic images...")

    for grade in CONFIG["target_grades"]:
        save_dir = os.path.join(CONFIG["output_dir"], "synthetic", f"grade_{grade}")
        os.makedirs(save_dir, exist_ok=True)

        n_batches = (CONFIG["n_generate"] + CONFIG["fid_batch"] - 1) // CONFIG["fid_batch"]
        generated = 0

        with torch.no_grad():
            for _ in tqdm(range(n_batches), desc=f"Grade {grade}"):
                bs     = min(CONFIG["fid_batch"], CONFIG["n_generate"] - generated)
                z      = torch.randn(bs, CONFIG["latent_dim"], device=device)
                labels = torch.full((bs,), grade, dtype=torch.long, device=device)
                imgs   = G(z, labels).cpu()
                imgs   = (imgs + 1) / 2   # → [0, 1]

                for i, img in enumerate(imgs):
                    vutils.save_image(img, os.path.join(save_dir, f"syn_{grade}_{generated+i:05d}.png"))
                generated += bs

        print(f"  Grade {grade}: {generated} images saved → {save_dir}")


generate_synthetic_images(G, CONFIG, device)

In [ ]:
# ── Visual inspection of generated images ───────────────────────────────────
# This is an informal but essential quality check before running FID.
# Do the images look like fundus photographs? Is there circular morphology?
# Are there any visible haemorrhages or lesions in Grade 3/4 samples?

fig, axes = plt.subplots(2, 8, figsize=(16, 5))
fig.suptitle("Generated Synthetic Fundus Images", fontsize=13)

for row_idx, grade in enumerate(CONFIG["target_grades"]):
    syn_dir = os.path.join(CONFIG["output_dir"], "synthetic", f"grade_{grade}")
    files   = sorted(os.listdir(syn_dir))[:8]
    for col_idx, fname in enumerate(files):
        img = Image.open(os.path.join(syn_dir, fname))
        axes[row_idx][col_idx].imshow(img)
        axes[row_idx][col_idx].axis("off")
        if col_idx == 0:
            axes[row_idx][col_idx].set_title(f"Grade {grade}", fontsize=10, loc="left")

plt.tight_layout()
plt.savefig(os.path.join(CONFIG["output_dir"], "synthetic_samples_grid.png"), dpi=150)
plt.show()

## 7. Evaluation — Fréchet Inception Distance (FID)

### Why FID?

Evaluating generative models is hard. The lecture covered this directly (slide 60): unlike discriminative models where we have a ground truth label to compare against, generative models can produce multiple valid outputs, and there's no single correct answer.

**FID** is the current standard metric (slide 61). It works by:
1. Extracting features from both real and generated images using a **pretrained Inception-v3** network
2. Fitting a Gaussian to each feature set: `N(μ_r, Σ_r)` for real, `N(μ_g, Σ_g)` for generated
3. Computing the **Wasserstein-2 (Fréchet) distance** between the two Gaussians:

$$\text{FID} = \|\mu_r - \mu_g\|^2 + \text{Tr}\left(\Sigma_r + \Sigma_g - 2(\Sigma_r \Sigma_g)^{1/2}\right)$$

**Lower FID = better** — the generated distribution is closer to the real distribution in feature space.

### FID Limitations (worth discussing in the report)

The lecture also noted (slide 61) that FID is **biased for small sample sizes** — you ideally need ~10,000 samples per distribution for reliable estimates. With only 500 generated images per grade, our FID numbers will have high variance. This is a legitimate limitation to acknowledge.

Additionally, FID uses features from Inception-v3 which was trained on ImageNet (natural images), not fundus photographs. This is the gap the lecture highlighted (slide 58): medical imaging evaluation requires both **perceptual** and **clinical** metrics. FID addresses perceptual similarity but cannot assess whether the generated retinal structures are clinically valid.

In [ ]:
def compute_fid(CONFIG, device):
    """
    Compute FID between real grade images and synthetic images.
    """
    from pytorch_fid import fid_score

    print("Computing FID scores...")
    fid_results = {}

    # Build image lookup once (train_1 ... train_5)
    image_lookup = {}
    for d in os.listdir(CONFIG["data_dir"]):
        folder = os.path.join(CONFIG["data_dir"], d)
        if d.startswith("train") and os.path.isdir(folder):
            for fname in os.listdir(folder):
                if fname.endswith(".jpeg"):
                    image_lookup[fname.replace(".jpeg", "")] = os.path.join(folder, fname)

    tf = transforms.Compose([
        transforms.Resize((CONFIG["img_size"], CONFIG["img_size"])),
        transforms.ToTensor(),
    ])

    for grade in CONFIG["target_grades"]:
        real_dir = os.path.join(CONFIG["data_dir"], "fid_real", f"grade_{grade}")
        fake_dir = os.path.join(CONFIG["output_dir"], "synthetic", f"grade_{grade}")

        # Prepare real images for FID (only if not already done)
        if not os.path.exists(real_dir) or len(os.listdir(real_dir)) == 0:
            print(f"  Preparing real grade {grade} images...")
            os.makedirs(real_dir, exist_ok=True)
            df  = pd.read_csv(os.path.join(CONFIG["data_dir"], "trainLabels.csv"))
            sub = df[df["level"] == grade]
            for _, row in tqdm(sub.iterrows(), total=len(sub)):
                src = image_lookup.get(row["image"])
                if src is None or not os.path.exists(src):
                    continue
                img = Image.open(src).convert("RGB")
                img = tf(img)
                vutils.save_image(img, os.path.join(real_dir, row["image"] + ".png"))

        n_real = len(os.listdir(real_dir))
        n_fake = len(os.listdir(fake_dir)) if os.path.exists(fake_dir) else 0
        print(f"  Real dir: {real_dir}, count: {n_real}")
        print(f"  Fake dir: {fake_dir}, count: {n_fake}")

        if n_real == 0 or n_fake == 0:
            print(f"  Skipping Grade {grade}: not enough images (real={n_real}, fake={n_fake})")
            continue

        # Cap batch_size to available images to avoid batch_size=0
        batch_size = min(CONFIG["fid_batch"], n_real, n_fake)

        print(f"  FID: Grade {grade} real vs synthetic...")
        fid_val = fid_score.calculate_fid_given_paths(
            [real_dir, fake_dir],
            batch_size=batch_size,
            device=device,
            dims=2048
        )
        fid_results[grade] = fid_val
        print(f"  Grade {grade} FID = {fid_val:.2f}")

    # Save
    with open(os.path.join(CONFIG["output_dir"], "fid_scores.txt"), "w") as f:
        f.write("FID Scores\n" + "="*30 + "\n")
        for g, v in fid_results.items():
            f.write(f"Grade {g}: {v:.4f}\n")

    return fid_results

fid_results = compute_fid(CONFIG, device)

In [ ]:
# ── FID result summary ──────────────────────────────────────────────────────

if fid_results:
    print("\nFID Score Summary")
    print("=" * 40)
    for grade, score in fid_results.items():
        if score < 20:
            quality = "good"
        elif score < 50:
            quality = "decent"
        else:
            quality = "poor — consider more epochs or larger model"
        print(f"  Grade {grade}: FID = {score:.2f}  [{quality}]")
    print()
    print("FID < 10 is excellent, 10-20 is good, 20-50 is acceptable")
    print("These thresholds are rough guides — medical image FID is not")
    print("directly comparable to natural image FID (different feature spaces).")

## 8. Downstream Classifier Comparison

### Utility-Based Evaluation

FID measures how similar the generated distribution is to the real one. But as the lecture noted (slide 64), **utility-based evaluation** asks a more clinically meaningful question: *do the generated images actually help a downstream task?*

We test this by training a **ResNet-18 classifier** twice:
- **(A) Real data only** — baseline, trained on the original imbalanced dataset
- **(B) Real + Synthetic** — same setup but with 500 Grade 3 and 500 Grade 4 images added

If the GAN-augmented classifier achieves higher validation accuracy (especially on minority grades), it demonstrates that our generated images have **clinical utility**, not just perceptual similarity.

This is precisely the data augmentation use case shown in slide 71.

In [ ]:
class DRClassifier(nn.Module):
    """
    ResNet-18 fine-tuned for 5-class DR grading.
    Using transfer learning from ImageNet — a pragmatic choice given the
    limited training data, especially for Grades 3 and 4.
    """
    def __init__(self, n_classes=5, pretrained=True):
        super().__init__()
        weights = models.ResNet18_Weights.DEFAULT if pretrained else None
        self.backbone = models.resnet18(weights=weights)
        self.backbone.fc = nn.Linear(self.backbone.fc.in_features, n_classes)

    def forward(self, x):
        return self.backbone(x)


class SyntheticAugmentedDataset(Dataset):
    """
    Wraps a real dataset and appends synthetic images from disk.
    The synthetic images are saved as PNG files from the generation step.
    """
    def __init__(self, real_dataset, synthetic_dirs, transform=None):
        self.real_dataset = real_dataset
        self.synthetic    = []
        for grade, syn_dir in synthetic_dirs.items():
            for fname in os.listdir(syn_dir):
                self.synthetic.append((os.path.join(syn_dir, fname), int(grade)))
        self.transform = transform
        print(f"Dataset: {len(real_dataset)} real + {len(self.synthetic)} synthetic images")

    def __len__(self):
        return len(self.real_dataset) + len(self.synthetic)

    def __getitem__(self, idx):
        if idx < len(self.real_dataset):
            return self.real_dataset[idx]
        img_path, label = self.synthetic[idx - len(self.real_dataset)]
        img = Image.open(img_path).convert("RGB")
        if self.transform:
            img = self.transform(img)
        return img, label

In [ ]:
def train_classifier(name, dataloader, val_loader, device, n_classes=5, epochs=15):
    model     = DRClassifier(n_classes=n_classes).to(device)
    optimizer = optim.Adam(model.parameters(), lr=1e-4)
    criterion = nn.CrossEntropyLoss()
    scheduler = optim.lr_scheduler.StepLR(optimizer, step_size=10, gamma=0.5)
    history   = []
    best_acc  = 0.0

    for epoch in range(1, epochs + 1):
        model.train()
        for imgs, labels in tqdm(dataloader, desc=f"[{name}] Epoch {epoch}", leave=False):
            imgs, labels = imgs.to(device), labels.to(device)
            optimizer.zero_grad()
            loss = criterion(model(imgs), labels)
            loss.backward()
            optimizer.step()
        scheduler.step()

        model.eval()
        correct = total = 0
        with torch.no_grad():
            for imgs, labels in val_loader:
                imgs, labels = imgs.to(device), labels.to(device)
                preds    = model(imgs).argmax(dim=1)
                correct += (preds == labels).sum().item()
                total   += labels.size(0)
        acc = correct / total
        history.append(acc)
        best_acc = max(best_acc, acc)
        print(f"  [{name}] Epoch {epoch:>3}  Val Acc: {acc:.4f}")

    return best_acc, history

In [ ]:
RUN_CLASSIFIER = True   # set to False to skip this section

if RUN_CLASSIFIER:
    print("Setting up classifier comparison experiment...")

    CLASSIFIER_SAMPLES_PER_CLASS = 200  # cap per grade for speed

    # ── Build a balanced 200-per-class subset from the full dataset ──────────
    full_ds = KaggleDRDataset(CONFIG["data_dir"], transform=eval_tf)
    all_labels = np.array(full_ds.labels)

    np.random.seed(SEED)
    selected_indices = []
    for grade in range(CONFIG["n_classes"]):
        grade_idx = np.where(all_labels == grade)[0]
        n = min(CLASSIFIER_SAMPLES_PER_CLASS, len(grade_idx))
        selected_indices.extend(np.random.choice(grade_idx, n, replace=False))

    selected_indices = np.array(selected_indices)
    np.random.shuffle(selected_indices)
    balanced_ds = Subset(full_ds, selected_indices)

    total = len(balanced_ds)
    n_val   = int(0.2 * total)
    n_train = total - n_val

    print(f"Balanced subset: {total} images ({CLASSIFIER_SAMPLES_PER_CLASS}/class), "
          f"train={n_train}, val={n_val}")

    train_ds, val_ds = torch.utils.data.random_split(
        balanced_ds, [n_train, n_val],
        generator=torch.Generator().manual_seed(SEED)
    )

    val_loader = DataLoader(val_ds, batch_size=32, shuffle=False,
                            num_workers=CONFIG["num_workers"])

    # (A) Baseline: real data only
    train_loader_real = DataLoader(train_ds, batch_size=32, shuffle=True,
                                   num_workers=CONFIG["num_workers"])
    acc_real, hist_real = train_classifier(
        "Real only", train_loader_real, val_loader, device, epochs=15
    )

    # (B) Augmented: real + synthetic Grade 3 & 4
    synthetic_dirs = {
        g: os.path.join(CONFIG["output_dir"], "synthetic", f"grade_{g}")
        for g in CONFIG["target_grades"]
        if os.path.exists(os.path.join(CONFIG["output_dir"], "synthetic", f"grade_{g}"))
    }

    if synthetic_dirs:
        aug_ds = SyntheticAugmentedDataset(train_ds, synthetic_dirs, transform=eval_tf)
        train_loader_aug = DataLoader(aug_ds, batch_size=32, shuffle=True,
                                      num_workers=CONFIG["num_workers"])
        acc_aug, hist_aug = train_classifier(
            "Real+Synthetic", train_loader_aug, val_loader, device, epochs=15
        )
    else:
        print("No synthetic images found — generate them first (Section 6).")

In [ ]:
if RUN_CLASSIFIER and synthetic_dirs:
    # ── Results comparison ──────────────────────────────────────────────────
    delta = acc_aug - acc_real
    direction = "improvement" if delta > 0 else "degradation"

    print(f"\n{'='*45}")
    print(f"  Real only          best val acc: {acc_real:.4f} ({acc_real*100:.1f}%)")
    print(f"  Real + Synthetic   best val acc: {acc_aug:.4f}  ({acc_aug*100:.1f}%)")
    print(f"  Delta:             {delta:+.4f} ({direction})")
    print(f"{'='*45}")

    # Visualise
    fig, ax = plt.subplots(figsize=(10, 4))
    ax.plot(hist_real, label="Real only",      marker="o", color="#E63946", linewidth=1.5)
    ax.plot(hist_aug,  label="Real+Synthetic", marker="s", color="#2A9D8F", linewidth=1.5)
    ax.set_xlabel("Epoch"); ax.set_ylabel("Validation Accuracy")
    ax.set_title("Classifier: Real Only vs GAN-Augmented Training")
    ax.legend(); ax.grid(alpha=0.3)
    ax.spines[["top", "right"]].set_visible(False)

    plt.tight_layout()
    plt.savefig(os.path.join(CONFIG["output_dir"], "classifier_comparison.png"), dpi=150)
    plt.show()

    # Save results
    with open(os.path.join(CONFIG["output_dir"], "classifier_results.txt"), "w") as f:
        f.write(f"Real only best acc:      {acc_real:.4f}\n")
        f.write(f"Real+Synthetic best acc: {acc_aug:.4f}\n")
        f.write(f"Delta:                   {delta:+.4f}\n")

## 9. Discussion

### Interpreting the Results

#### FID Score
A lower FID suggests that the generated fundus images lie closer to the real data distribution in Inception feature space. However, as noted in the lecture (slide 65), *no single metric is perfect*. FID captures perceptual similarity but not clinical validity — a generated image could have a realistic circular fundus shape but anatomically incorrect lesion patterns.

#### Classifier Comparison
- **If accuracy improved**: the synthetic Grade 3/4 images provided useful signal that helped the classifier generalise to underrepresented classes. This validates our core hypothesis.
- **If accuracy did not improve (or degraded)**: this is a valid negative result. Possible explanations:
  - The GAN hadn't trained long enough — the synthetic images were too blurry or generic to be informative (recall: GANs are unstable, from slide 42)
  - Mode collapse occurred — the GAN generated a limited set of images that don't cover the diversity of real Grade 3/4 fundus patterns
  - 500 synthetic images is a small augmentation relative to the size of the real training set

### Limitations & Future Work

1. **Compute constraints**: 128px images and 100 epochs is a starting point. StyleGAN or a diffusion model would produce higher-quality images but require significantly more GPU time.
2. **FID sample size**: the lecture noted FID needs ~10k samples to be reliable (slide 61). Our 500 generated images per grade will give high-variance FID estimates.
3. **Clinical evaluation**: the gap between general vision and biomedical imaging (slide 58) means we really need a clinician to assess whether the synthetic images contain the correct pathological features (haemorrhages, neovascularisation) for Grades 3 and 4.
4. **Diffusion as future direction**: as discussed in the lecture, diffusion models achieve higher image quality than GANs and have more stable training. A natural extension of this project would be to implement a conditional DDPM and compare quality with our cGAN.

### Connection to the Lecture Taxonomy

| Concept from Lecture | Where it appears in this project |
|---|---|
| **Data Problem** (slide 10) | Core motivation — Grade 3/4 under-representation |
| **Implicit Density / Adversarial** (slide 21) | Our GAN architecture |
| **Conditional Generation p(x\|c)** (slide 16, 53) | Grade label as control signal |
| **cGAN objective** (slide 55) | Implemented in `train_gan()` |
| **GAN: sharp but unstable vs VAE: blurry but stable** (slide 42) | Discussed in model choice rationale |
| **Input Concatenation conditioning** (slide 57) | How grade label enters G and D |
| **FID metric** (slide 61) | Primary evaluation metric |
| **Utility-based evaluation** (slide 64) | Classifier comparison experiment |
| **Data Augmentation application** (slide 71) | The overall framing of our project |